# 00 · Sanity Check

Verify the three core ML libraries work end-to-end on a single audio sample. If any cell fails here, fail **now**, not in week 3.

**Prerequisites before running:**
1. `venv` is active (`source venv/bin/activate` from the project root).
2. The VS Code kernel for this notebook is set to the project venv — pick it from the kernel selector in the top-right of this notebook.
3. A test WAV file exists at `../data/samples/test.wav`. If it doesn't, the first cell will guide you to create one.

**What this notebook checks (in order):**
1. PRAAT / Parselmouth — pitch + intensity extraction
2. OpenAI Whisper — speech-to-text transcription
3. wav2vec2 — emotion classification

First run will download ~500 MB of model weights (Whisper base + wav2vec2). Be patient.

## 0 · Setup & locate the sample

In [ ]:
import os

SAMPLE = "../data/samples/test.wav"

if not os.path.exists(SAMPLE):
    raise FileNotFoundError(
        f"\nMissing {SAMPLE}.\n"
        "Generate one with macOS built-in TTS (run this in your terminal, from project root):\n\n"
        "  say -o data/samples/test.aiff \"I'm fine, really, I'm fine\"\n"
        "  ffmpeg -y -i data/samples/test.aiff -ar 16000 -ac 1 data/samples/test.wav\n\n"
        "Then re-run this cell."
    )

size_kb = os.path.getsize(SAMPLE) / 1024
print(f"Sample found: {SAMPLE} ({size_kb:.1f} KB)")

## 1 · PRAAT / Parselmouth — prosody extraction

Tests that the PRAAT phonetics engine works through its Python binding. Extracts mean pitch (F0) and mean intensity.

In [ ]:
import parselmouth
from parselmouth.praat import call

snd = parselmouth.Sound(SAMPLE)
print(f"Duration:       {snd.duration:.2f} s")
print(f"Sample rate:    {int(snd.sampling_frequency)} Hz")
print(f"Num channels:   {snd.n_channels}")

pitch = snd.to_pitch()
intensity = snd.to_intensity()

mean_f0  = call(pitch, "Get mean", 0, 0, "Hertz")
stdev_f0 = call(pitch, "Get standard deviation", 0, 0, "Hertz")
mean_int = call(intensity, "Get mean", 0, 0, "energy")

print(f"\nMean F0 (pitch):    {mean_f0:.1f} Hz")
print(f"F0 std deviation:   {stdev_f0:.1f} Hz")
print(f"Mean intensity:     {mean_int:.1f}")

print("\n[OK] Parselmouth working.")

## 2 · OpenAI Whisper — transcription

Loads the **base** model (74 MB, fast) just for the sanity check. In production we'll switch to `large-v3-turbo`.

In [ ]:
import whisper

print("Loading Whisper base model (first run downloads ~74MB)...")
model = whisper.load_model("base")
print("Model loaded. Transcribing...")

result = model.transcribe(SAMPLE)

print(f"\nDetected language: {result['language']}")
print(f"Transcription:     {result['text'].strip()}")

print("\n[OK] Whisper working.")

## 3 · wav2vec2 — emotion classification

Uses `superb/wav2vec2-base-superb-er` — pretrained on IEMOCAP, outputs four classes: **ang** (anger), **hap** (happiness), **neu** (neutral), **sad** (sadness).

In [ ]:
from transformers import pipeline

print("Loading wav2vec2 emotion classifier (first run downloads ~360MB)...")
emo = pipeline("audio-classification", model="superb/wav2vec2-base-superb-er")
print("Model loaded. Classifying...")

predictions = emo(SAMPLE)

print("\nEmotion predictions (sorted by confidence):")
for p in predictions:
    bar = "█" * int(p["score"] * 40)
    print(f"  {p['label']:>4}: {p['score']:.3f}  {bar}")

print("\n[OK] wav2vec2 working.")

## Done

If all three sections printed `[OK]` — the entire ML pipeline foundation works. We can proceed to Phase 1 (Tiny Visual Win) with confidence.